# Why does SMA200 / monthly timing improve risk-adjusted performance?

This notebook validates the **SMA200, monthly-rebalanced** SPY / T-bill rule from `sma_trend_backtest.ipynb`.
It does not tune it: **`SMA_WINDOW = 200` and monthly rebalancing are fixed throughout.**

Questions:

1. Does the regime at a decision date (close above / below its 200-day SMA) carry information about the **next month's** equity excess return over T-bills?
   - H1: $E[R_{SPY} - R_{Tbill} \mid P > SMA_{200}] > 0$
   - H2: $E[R_{SPY} - R_{Tbill} \mid P < SMA_{200}] < 0$
2. Is that statistically significant (iid t-test, Newey-West, block bootstrap)?
3. Is it stable across sub-periods back to the 1930s?
4. Is the edge **(A)** a genuine shift in expected excess returns, or **(B)** mainly the avoidance of left-tail events and long bear markets?
5. Where does the higher Sharpe ratio come from (return, volatility, drawdowns, T-bill income)?

## Timing (no look-ahead)

| Step | SPY sample (1993+) -- identical to the backtest | Long S&P 500 sample (1928+) |
|---|---|---|
| Decision date $d_m$ | last trading day of month *m* | same |
| Signal | $\mathbb{1}[P_{d_m} > SMA_{200,d_m}]$, SPY split-adjusted close; uses closes $\le d_m$ only | same, with the `^GSPC` price-index close |
| Fill | **open** of the next trading day $d_m+1$ | **close** of the next trading day $d_m+1$ (daily opens before 1962 are unreliable) |
| Evaluated return | total return from fill $m$ to fill $m+1$ (open-to-open) | total return from fill $m$ to fill $m+1$ (close-to-close) |
| T-bill return | `CASH_SOURCE` (default `bil`: BIL ETF 2007+, `^IRX` accrual before) over the same interval | 3-month T-bill yield known when the interval starts, accrued over its calendar days |

So the return used to evaluate each signal starts strictly **after** the signal is observed: at least one trading day later, which also includes the overnight gap.
The monthly holding periods are exactly the strategy's holding periods, so the regime statistics and the backtest describe the same returns.

> Research / decision-support only. No orders are placed. Past performance is not indicative of future results.

In [1]:
# ---- Parameters (SMA window and rebalancing are FIXED for this study) --------
SMA_WINDOW = 200
REBALANCE = "monthly"
COST_BPS = 5.0           # per switch, as in the backtest
CASH_SOURCE = "bil"      # SPY sample: "irx" | "bil" | "zero" (as in the backtest)
USER_START = "2007-03-15"  # your backtest window, used for the reproduction check
LONG_START = "1934-01-01"  # long sample: first decision date with T-bill data (FRED TB3MS starts 1934)
N_BOOT = 10_000          # bootstrap replications
BLOCK = 6                # bootstrap block length in months (keeps regime persistence / vol clustering)
NW_LAGS = 6              # Newey-West lags (months)
SEED = 42
REFRESH_DATA = False
CACHE_MAX_AGE_HOURS = 12

SHILLER_URLS = [
    "https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/ie_data.xls",
    "http://www.econ.yale.edu/~shiller/data/ie_data.xls",
]
FRED_TB3MS_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=TB3MS"

In [2]:
import math
import sys
import time
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy import stats

# Shared helpers: notebooks/backtest/bt_common.py
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
from bt_common import (  # noqa: E402
    CACHE_DIR, EXECUTIONS, REBALANCE_FREQ, cash_returns, check_days, decision_panel, pct, setup_plotly, sma_backtest,
    total_return_frame,
)
from bt_common import block_indices as _block_indices, fetch_file as _fetch_file  # noqa: E402
from bt_common import load_history as _load_history, newey_west as _newey_west  # noqa: E402

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
C_ON, C_OFF, C_BH, C_STRAT, C_MIX = "#2ca02c", "#d62728", "#ff7f0e", "#1f77b4", "#7f7f7f"

# fig.show() emits plain HTML, so charts render in VS Code without the "Jupyter Notebook Renderers" extension.
PLOTLY_JS = "cdn"  # "cdn" | "inline" (offline, ~4.6 MB per chart)
setup_plotly(PLOTLY_JS)
load_history = partial(_load_history, refresh=REFRESH_DATA, max_age_hours=CACHE_MAX_AGE_HOURS)
fetch_file = partial(_fetch_file, refresh=REFRESH_DATA)

## 0. Data

| Series | Source | Used for |
|---|---|---|
| SPY daily OHLC + `Adj Close` | Yahoo Finance (`yfinance`) | SPY sample, identical to the backtest |
| BIL daily, `^IRX` daily (13-week T-bill discount yield, 1960+) | Yahoo Finance | T-bills, SPY sample |
| `^GSPC` daily close (1928+) | Yahoo Finance | long-sample **signal** (price index). Before March 1957 this is the S&P 90 daily composite, the predecessor of the S&P 500 |
| S&P 500 dividends, monthly (annualised rate, 1871 - mid-2024) | Robert Shiller, *Irrational Exuberance* data (`ie_data.xls`) | long-sample **total return** before 1988 |
| `^SP500TR` daily (1988+) | Yahoo Finance (official S&P 500 Total Return index) | long-sample total return from 1988 |
| `TB3MS` monthly (1934+) | FRED, 3-month T-bill secondary-market rate | T-bill yield before 1960 |

**Assumptions for the long S&P 500 total-return series**

* 1928-1987: daily return $= (P_t + D_m/252)/P_{t-1} - 1$, where $D_m$ is Shiller's annualised dividend for the month. Dividends accrue evenly over trading days and are reinvested daily. A validation against the official `^SP500TR` over 1989-2023 is shown below.
* 1988 onwards: the official `^SP500TR` index.
* T-bill yield: `^IRX` close from 1960. Before that, FRED `TB3MS` for month *m*, which is only treated as known from the first day of month *m+1*. A holding period accrues the yield known when it starts: $(1+y/100)^{days/365}-1$.
* No costs, taxes or fund fees on the index. SPY's returns already include its expense ratio.

In [3]:
# --- SPY sample (exactly as in the backtest notebook) ---
px_spy = total_return_frame(load_history("SPY"))
px_spy = px_spy[px_spy["open"] > 0]
irx = load_history("^IRX")["Close"].rename("irx_pct")
bil_px = total_return_frame(load_history("BIL"))


def cash_returns_spy(index: pd.DatetimeIndex, field: str) -> pd.Series:
    return cash_returns(index, field, CASH_SOURCE, irx, bil_px)


cash_spy = {f: cash_returns_spy(px_spy.index, f) for f in ("tr_open", "tr_close")}

# --- Long S&P 500 sample ---
gspc = load_history("^GSPC")["Close"]
sptr = load_history("^SP500TR")["Close"]

shiller_path = fetch_file(SHILLER_URLS, CACHE_DIR / "shiller_ie_data.xls")
sh = pd.read_excel(shiller_path, sheet_name="Data", header=None, skiprows=8, usecols=[0, 1, 2])
sh.columns = ["date", "P", "D"]
sh = sh[pd.to_numeric(sh["date"], errors="coerce").notna() & pd.to_numeric(sh["D"], errors="coerce").notna()]
year = sh["date"].astype(float).astype(int)
month = ((sh["date"].astype(float) - year) * 100).round().astype(int)
sh.index = pd.PeriodIndex.from_fields(year=year, month=month, freq="M")
div_annual = sh["D"].astype(float)

tb_path = fetch_file([FRED_TB3MS_URL], CACHE_DIR / "fred_TB3MS.csv", max_age_days=7)
tb = pd.read_csv(tb_path)
tb_known = pd.Series(
    tb.iloc[:, 1].astype(float).values,
    index=pd.to_datetime(tb.iloc[:, 0]) + pd.offsets.MonthBegin(1),  # month m average, known from month m+1
)

# Synthetic total return 1928-1987 (price + Shiller dividends), official ^SP500TR from 1988.
d_daily = div_annual.reindex(gspc.index.to_period("M")).ffill().to_numpy()
r_syn = ((gspc + d_daily / 252.0) / gspc.shift(1) - 1).rename("r_syn")
r_off = sptr.pct_change(fill_method=None).reindex(gspc.index)
splice = sptr.index[1]
r_tr = r_syn.where(gspc.index < splice, r_off).dropna()

px_idx = pd.DataFrame({"close": gspc, "tr_close": (1 + r_tr).cumprod()}).dropna()
px_idx["tr_close"] = px_idx["tr_close"] / px_idx["tr_close"].iloc[0]

yld_long = pd.concat([tb_known[tb_known.index < irx.index[0]], irx]).sort_index()
yld_long = yld_long[~yld_long.index.duplicated(keep="last")]


def cash_returns_long(index: pd.DatetimeIndex) -> pd.Series:
    days = pd.Series(index, index=index).diff().dt.days
    y = yld_long.reindex(index.union(yld_long.index)).ffill().reindex(index).shift(1)
    return (1 + y / 100.0) ** (days / 365.0) - 1


cash_idx = cash_returns_long(px_idx.index)

print(f"SPY       {px_spy.index[0].date()} -> {px_spy.index[-1].date()}")
print(f"^GSPC     {gspc.index[0].date()} -> {gspc.index[-1].date()}  (signal)")
print(f"S&P TR    synthetic until {splice.date()}, ^SP500TR after; Shiller dividends to {div_annual.index[-1]}")
print(f"T-bills   TB3MS from {tb_known.index[0].date()}, ^IRX from {irx.index[0].date()}")

SPY       1993-01-29 -> 2026-10-19
^GSPC     1927-12-30 -> 2026-10-19  (signal)
S&P TR    synthetic until 1988-01-05, ^SP500TR after; Shiller dividends to 2024-06
T-bills   TB3MS from 1934-02-01, ^IRX from 1960-01-04


In [4]:
# Validation of the synthetic total-return construction against the official index (overlap 1989-2023).
syn_y = (1 + r_syn).groupby(r_syn.index.year).prod() - 1
off_y = (1 + r_off).groupby(r_off.index.year).prod() - 1
val = pd.DataFrame({"synthetic": syn_y, "^SP500TR": off_y}).loc[1989:2023].dropna()
val["diff"] = val["synthetic"] - val["^SP500TR"]
print(f"Synthetic vs official S&P 500 TR, {val.index[0]}-{val.index[-1]}: mean annual difference "
      f"{val['diff'].mean():+.2%}, tracking error {val['diff'].std():.2%}, max |diff| {val['diff'].abs().max():.2%}")

Synthetic vs official S&P 500 TR, 1989-2023: mean annual difference -0.06%, tracking error 0.10%, max |diff| 0.40%


## 0b. Backtest engine (same as the backtest notebook) and reproduction check

`backtest()` is the shared engine `sma_backtest` from `notebooks/backtest/bt_common.py`, the same code the backtest notebook runs, with the cash series passed in. The check below should reproduce the notebook's SMA200 / monthly figures for your window (`START = 2007-03-15`, `bil`, 5 bp).

In [5]:
backtest = sma_backtest  # shared engine (bt_common); every call below passes rebalance explicitly


def perf_stats(ret, rf, pos=None, turnover=None, periods=252) -> dict:
    ret = ret.dropna()
    rf = rf.reindex(ret.index).fillna(0.0)
    eq = (1 + ret).cumprod()
    years = (ret.index[-1] - ret.index[0]).days / 365.25
    cagr = eq.iloc[-1] ** (1 / years) - 1
    ex = ret - rf
    downside = math.sqrt((np.minimum(ex, 0) ** 2).mean()) * math.sqrt(periods)
    max_dd = (eq / eq.cummax() - 1).min()
    out = {
        "CAGR": cagr,
        "Arithmetic return (ann.)": ret.mean() * periods,
        "Excess return (ann.)": ex.mean() * periods,
        "Volatility (ann.)": ret.std() * math.sqrt(periods),
        "Sharpe": ex.mean() / ex.std() * math.sqrt(periods) if ex.std() > 0 else np.nan,
        "Sortino": ex.mean() * periods / downside if downside > 0 else np.nan,
        "Max drawdown": max_dd,
        "Calmar": cagr / abs(max_dd) if max_dd < -1e-3 else np.nan,
    }
    if pos is not None:
        out["% time in equities"] = pos.reindex(ret.index).mean()
        out["% time in T-bills"] = 1 - out["% time in equities"]
    if turnover is not None:
        sw = int((turnover.reindex(ret.index) > 0).sum())
        out["Switches"] = sw
        out["Switches / yr"] = sw / years
    return out


chk = backtest(px_spy, cash_spy["tr_open"], SMA_WINDOW, COST_BPS, "next_open", start=USER_START, rebalance=REBALANCE)
s = perf_stats(chk["r_strat"], chk["r_cash"], chk["pos"], chk["turnover"])
print(f"Reproduction, SMA{SMA_WINDOW}/{REBALANCE}, {chk.index[0].date()} -> {chk.index[-1].date()}, cash={CASH_SOURCE}: "
      f"CAGR {s['CAGR']:.2%}, Sharpe {s['Sharpe']:.2f}, max DD {s['Max drawdown']:.1%}, {s['Switches / yr']:.1f} switches/yr")
print("Backtest notebook (window sweep, same settings): CAGR 9.62%, Sharpe 0.69, max DD -25.8%, 1.5 switches/yr")

Reproduction, SMA200/monthly, 2007-03-15 -> 2026-10-19, cash=bil: CAGR 9.62%, Sharpe 0.69, max DD -25.8%, 1.5 switches/yr
Backtest notebook (window sweep, same settings): CAGR 9.62%, Sharpe 0.69, max DD -25.8%, 1.5 switches/yr


## 0c. Monthly decision panel

Each row is one monthly decision: the regime observed at the close of decision date $d_m$, then the SPY / S&P 500 total return, the T-bill return and the excess return over the **holding period that follows** (fill $m$ to fill $m+1$).
The panel is cut directly from the daily backtest rows, so its periods are exactly the strategy's holding periods.

In [6]:
def monthly_panel(bt: pd.DataFrame, lag: int) -> pd.DataFrame:
    # Shared decision panel (bt_common) plus the distance from the SMA and the regime label.
    out = decision_panel(bt, lag, extra={"dist": bt["close"] / bt["sma"] - 1})
    out["regime"] = np.where(out["state"] == 1, "Risk-on (P > SMA200)", "Risk-off (P < SMA200)")
    return out


bt_spy = backtest(px_spy, cash_spy["tr_open"], SMA_WINDOW, COST_BPS, "next_open", rebalance=REBALANCE)
bt_idx = backtest(px_idx, cash_idx, SMA_WINDOW, COST_BPS, "next_close", rebalance=REBALANCE)
pan_spy = monthly_panel(bt_spy, EXECUTIONS["next_open"][1])
pan_idx = monthly_panel(bt_idx, EXECUTIONS["next_close"][1]).loc[LONG_START:]

print(f"SPY panel:     {len(pan_spy)} decisions, {pan_spy.index[0].date()} -> {pan_spy.index[-1].date()}")
print(f"S&P 500 panel: {len(pan_idx)} decisions, {pan_idx.index[0].date()} -> {pan_idx.index[-1].date()}")
display(pan_spy.tail(3))

SPY panel:     393 decisions, 1993-12-31 -> 2026-08-31
S&P 500 panel: 1110 decisions, 1934-03-29 -> 2026-08-31


,hold_from,hold_to,state,dist,r_eq,r_tbill,excess,regime
decision,,,,,,,,
2026-06-30,2026-07-02,2026-08-03,1.0,0.080044,0.005960,0.003206,0.002754,Risk-on (P > SMA200)
2026-07-31,2026-08-04,2026-09-01,1.0,0.066597,0.016773,0.002845,0.013928,Risk-on (P > SMA200)
2026-08-31,2026-09-02,2026-10-01,1.0,0.079947,0.005575,0.002988,0.002587,Risk-on (P > SMA200)


In [7]:
REGIMES = ["Risk-on (P > SMA200)", "Risk-off (P < SMA200)"]
RNG = np.random.default_rng(SEED)


# Moving-block bootstrap and Newey-West HAC from bt_common, with this study's settings.
block_indices = partial(_block_indices, n_boot=N_BOOT, block=BLOCK, rng=RNG)
newey_west = partial(_newey_west, lags=NW_LAGS)


def regime_summary(pan: pd.DataFrame, eq_label: str) -> pd.DataFrame:
    rows = {}
    for name, g in [*((r, pan[pan["regime"] == r]) for r in REGIMES), ("All months", pan)]:
        rows[name] = {
            f"Mean {eq_label} return": g["r_eq"].mean(),
            f"Median {eq_label} return": g["r_eq"].median(),
            "Mean T-bill return": g["r_tbill"].mean(),
            "Mean excess return": g["excess"].mean(),
            "Mean excess (annualised x12)": g["excess"].mean() * 12,
            f"Volatility of {eq_label} (monthly)": g["r_eq"].std(),
            f"Volatility of {eq_label} (ann.)": g["r_eq"].std() * math.sqrt(12),
            "Observations (months)": len(g),
            f"% months {eq_label} beats T-bills": (g["excess"] > 0).mean(),
        }
    return pd.DataFrame(rows)


def show_summary(tab: pd.DataFrame, title: str):
    fmt = tab.astype(object).copy()
    for r in tab.index:
        fmt.loc[r] = [f"{int(v)}" if r.startswith("Observations") else pct(v) for v in tab.loc[r]]
    print(title)
    display(fmt)


def regime_inference(pan: pd.DataFrame) -> pd.DataFrame:
    ex = pan["excess"].to_numpy()
    on = (pan["state"] == 1).to_numpy()
    idx = block_indices(len(pan))
    ex_b, on_b = ex[idx], on[idx]
    boot_on = (ex_b * on_b).sum(1) / on_b.sum(1)
    boot_off = (ex_b * ~on_b).sum(1) / (~on_b).sum(1)
    beta_nw, se_nw = newey_west(ex, np.column_stack([on, ~on]).astype(float))
    beta_d, se_d = newey_west(ex, np.column_stack([np.ones_like(ex), on]).astype(float))
    rows = {}
    for k, (name, mask, boot, alt) in enumerate([(REGIMES[0], on, boot_on, "greater"), (REGIMES[1], ~on, boot_off, "less")]):
        x = ex[mask]
        n, m, se = len(x), x.mean(), x.std(ddof=1) / math.sqrt(len(x))
        tq = stats.t.ppf(0.975, n - 1)
        t_iid = m / se
        rows[name] = {
            "H1": "mean > 0" if alt == "greater" else "mean < 0",
            "n": n, "Mean excess (monthly)": m, "Std. error (iid)": se, "t-stat (iid)": t_iid,
            "95% CI (t)": f"[{m - tq * se:+.2%}, {m + tq * se:+.2%}]",
            "One-sided p (t)": stats.t.sf(t_iid, n - 1) if alt == "greater" else stats.t.cdf(t_iid, n - 1),
            "Std. error (Newey-West)": se_nw[k], "t-stat (Newey-West)": beta_nw[k] / se_nw[k],
            "95% CI (block bootstrap)": f"[{np.quantile(boot, 0.025):+.2%}, {np.quantile(boot, 0.975):+.2%}]",
            "One-sided p (bootstrap)": (boot <= 0).mean() if alt == "greater" else (boot >= 0).mean(),
        }
    diff_boot = boot_on - boot_off
    x_on, x_off = ex[on], ex[~on]
    w = stats.ttest_ind(x_on, x_off, equal_var=False, alternative="greater")
    se_w = math.sqrt(x_on.var(ddof=1) / len(x_on) + x_off.var(ddof=1) / len(x_off))
    d = x_on.mean() - x_off.mean()
    rows["Difference (on - off)"] = {
        "H1": "on > off", "n": len(ex), "Mean excess (monthly)": d, "Std. error (iid)": se_w, "t-stat (iid)": w.statistic,
        "95% CI (t)": f"[{d - 1.96 * se_w:+.2%}, {d + 1.96 * se_w:+.2%}]", "One-sided p (t)": w.pvalue,
        "Std. error (Newey-West)": se_d[1], "t-stat (Newey-West)": beta_d[1] / se_d[1],
        "95% CI (block bootstrap)": f"[{np.quantile(diff_boot, 0.025):+.2%}, {np.quantile(diff_boot, 0.975):+.2%}]",
        "One-sided p (bootstrap)": (diff_boot <= 0).mean(),
    }
    out = pd.DataFrame(rows).T
    out.attrs["boot"] = {"on": boot_on, "off": boot_off, "diff": diff_boot}
    return out


def show_inference(inf: pd.DataFrame, title: str):
    print(title)
    display(inf.style.format({
        "Mean excess (monthly)": "{:+.2%}", "Std. error (iid)": "{:.2%}", "t-stat (iid)": "{:+.2f}",
        "One-sided p (t)": "{:.3f}", "Std. error (Newey-West)": "{:.2%}", "t-stat (Newey-West)": "{:+.2f}",
        "One-sided p (bootstrap)": "{:.3f}",
    }))

## 1. Does the SMA200 regime predict next-month excess returns?

Regime at decision date $d_m$, returns over the following holding period (one month).

In [8]:
sum_spy = regime_summary(pan_spy, "SPY")
sum_idx = regime_summary(pan_idx, "S&P 500")
show_summary(sum_spy, f"SPY sample, {pan_spy.index[0]:%Y-%m} -> {pan_spy.index[-1]:%Y-%m} (next-open execution, cash={CASH_SOURCE})")
show_summary(sum_idx, f"Long S&P 500 TR sample, {pan_idx.index[0]:%Y-%m} -> {pan_idx.index[-1]:%Y-%m} (next-close execution, 3m T-bill)")

SPY sample, 1993-12 -> 2026-08 (next-open execution, cash=bil)


,Risk-on (P > SMA200),Risk-off (P < SMA200),All months
Mean SPY return,1.11%,0.53%,0.96%
Median SPY return,1.54%,1.10%,1.46%
Mean T-bill return,0.21%,0.17%,0.20%
Mean excess return,0.90%,0.36%,0.76%
Mean excess (annualised x12),10.77%,4.34%,9.17%
Volatility of SPY (monthly),3.49%,6.51%,4.44%
Volatility of SPY (ann.),12.11%,22.55%,15.38%
Observations (months),295,98,393
% months SPY beats T-bills,68.47%,60.20%,66.41%


Long S&P 500 TR sample, 1934-03 -> 2026-08 (next-close execution, 3m T-bill)


,Risk-on (P > SMA200),Risk-off (P < SMA200),All months
Mean S&P 500 return,1.18%,0.56%,0.99%
Median S&P 500 return,1.48%,0.89%,1.34%
Mean T-bill return,0.27%,0.30%,0.28%
Mean excess return,0.91%,0.27%,0.71%
Mean excess (annualised x12),10.93%,3.18%,8.53%
Volatility of S&P 500 (monthly),3.72%,5.95%,4.54%
Volatility of S&P 500 (ann.),12.89%,20.61%,15.72%
Observations (months),766,344,1110
% months S&P 500 beats T-bills,63.84%,52.62%,60.36%


## 2. Statistical significance

* **iid**: classic standard error $s/\sqrt{n}$, t-test and t-based 95% CI. The difference row uses Welch's unequal-variance t-test.
* **Newey-West**: regression of the excess return on regime dummies with HAC standard errors (6 lags), robust to autocorrelation and heteroskedasticity.
* **Block bootstrap**: a moving-block bootstrap of the monthly panel (blocks of 6 months, 10,000 draws). It makes no normality assumption and keeps regime persistence and volatility clustering. The regime mean is recomputed in every draw.
* One-sided p-values test **H1** (risk-on mean > 0, risk-off mean < 0, on > off).

In [9]:
inf_spy = regime_inference(pan_spy)
inf_idx = regime_inference(pan_idx)
show_inference(inf_spy, "SPY sample")
show_inference(inf_idx, "Long S&P 500 TR sample")

SPY sample


,H1,n,Mean excess (monthly),Std. error (iid),t-stat (iid),95% CI (t),One-sided p (t),Std. error (Newey-West),t-stat (Newey-West),95% CI (block bootstrap),One-sided p (bootstrap)
Risk-on (P > SMA200),mean > 0,295,+0.90%,0.20%,+4.40,"[+0.50%, +1.30%]",0.000,0.18%,+4.97,"[+0.54%, +1.27%]",0.000
Risk-off (P < SMA200),mean < 0,98,+0.36%,0.66%,+0.55,"[-0.95%, +1.67%]",0.708,0.67%,+0.54,"[-0.94%, +1.73%]",0.719
Difference (on - off),on > off,393,+0.54%,0.69%,+0.78,"[-0.82%, +1.89%]",0.219,0.70%,+0.77,"[-0.86%, +1.94%]",0.229


Long S&P 500 TR sample


,H1,n,Mean excess (monthly),Std. error (iid),t-stat (iid),95% CI (t),One-sided p (t),Std. error (Newey-West),t-stat (Newey-West),95% CI (block bootstrap),One-sided p (bootstrap)
Risk-on (P > SMA200),mean > 0,766,+0.91%,0.14%,+6.74,"[+0.65%, +1.18%]",0.000,0.14%,+6.72,"[+0.65%, +1.18%]",0.000
Risk-off (P < SMA200),mean < 0,344,+0.27%,0.32%,+0.83,"[-0.37%, +0.90%]",0.795,0.29%,+0.90,"[-0.28%, +0.88%]",0.842
Difference (on - off),on > off,1110,+0.65%,0.35%,+1.85,"[-0.04%, +1.33%]",0.032,0.32%,+2.01,"[-0.02%, +1.25%]",0.028


In [10]:
def ci_bar(inf, label, color_on=C_ON, color_off=C_OFF):
    b = inf.attrs["boot"]
    means = [inf.loc[REGIMES[0], "Mean excess (monthly)"], inf.loc[REGIMES[1], "Mean excess (monthly)"]]
    lo = [np.quantile(b["on"], 0.025), np.quantile(b["off"], 0.025)]
    hi = [np.quantile(b["on"], 0.975), np.quantile(b["off"], 0.975)]
    return go.Bar(
        x=[f"{label}<br>risk-on", f"{label}<br>risk-off"], y=means, marker_color=[color_on, color_off], name=label,
        error_y=dict(type="data", symmetric=False, array=np.subtract(hi, means), arrayminus=np.subtract(means, lo)),
        text=[f"{m:+.2%}" for m in means], textposition="outside", showlegend=False,
    )


fig = go.Figure([ci_bar(inf_spy, "SPY 1993+"), ci_bar(inf_idx, f"S&P 500 {pan_idx.index[0]:%Y}+")])
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(title="Mean next-month excess return over T-bills by regime (95% block-bootstrap CI)",
                  yaxis_tickformat=".1%", height=450)
fig.show()

fig = go.Figure()
for reg, col in zip(REGIMES, [C_ON, C_OFF]):
    for lab, pan in (("SPY", pan_spy), ("S&P 500", pan_idx)):
        x = pan.loc[pan["regime"] == reg, "excess"]
        fig.add_trace(go.Violin(x=[f"{lab}<br>{reg.split(' ')[0]}"] * len(x), y=x, line_color=col, box_visible=True,
                                meanline_visible=True, points="outliers", name=f"{lab} {reg}", showlegend=False))
fig.update_layout(title="Distribution of next-month excess returns by regime (box = quartiles, dashed = mean)",
                  yaxis_tickformat=".0%", height=520)
fig.show()

### Tails and volatility by regime

If the risk-off regime mainly carries **higher volatility and fatter left tails** rather than a reliably negative mean, the strategy's value comes from risk reduction (hypothesis B).

In [11]:
def tail_table(pan: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for reg in REGIMES:
        x = pan.loc[pan["regime"] == reg, "excess"]
        q5 = x.quantile(0.05)
        rows[reg] = {
            "n": len(x), "Mean": x.mean(), "Median": x.median(), "Volatility (monthly)": x.std(),
            "Skewness": stats.skew(x), "Excess kurtosis": stats.kurtosis(x),
            "5% quantile": q5, "CVaR 5% (mean of worst 5%)": x[x <= q5].mean(), "Worst month": x.min(),
            "% months < -5%": (x < -0.05).mean(), "% months < -10%": (x < -0.10).mean(),
            "Sharpe of the regime (ann.)": x.mean() / x.std() * math.sqrt(12),
        }
    return pd.DataFrame(rows)


def vol_ratio_ci(pan: pd.DataFrame):
    ex, on = pan["excess"].to_numpy(), (pan["state"] == 1).to_numpy()
    idx = block_indices(len(pan))
    ex_b, on_b = ex[idx], on[idx]
    sd = lambda m: np.sqrt((np.where(m, ex_b, 0) ** 2).sum(1) / m.sum(1) - ((np.where(m, ex_b, 0)).sum(1) / m.sum(1)) ** 2)
    ratio = sd(~on_b) / sd(on_b)
    return ex[~on].std() / ex[on].std(), np.quantile(ratio, [0.025, 0.975])


TAIL_FMT = {c: "{:+.2%}" for c in ["Mean", "Median", "5% quantile", "CVaR 5% (mean of worst 5%)", "Worst month"]}
TAIL_FMT.update({"Volatility (monthly)": "{:.2%}", "% months < -5%": "{:.1%}", "% months < -10%": "{:.1%}",
                 "Skewness": "{:+.2f}", "Excess kurtosis": "{:.2f}", "Sharpe of the regime (ann.)": "{:+.2f}", "n": "{:.0f}"})
for lab, pan in (("SPY sample", pan_spy), ("Long S&P 500 TR sample", pan_idx)):
    print(lab)
    display(tail_table(pan).T.style.format(TAIL_FMT))
    r, (lo, hi) = vol_ratio_ci(pan)
    print(f"  volatility ratio risk-off / risk-on = {r:.2f}  (95% block-bootstrap CI {lo:.2f} - {hi:.2f})\n")

SPY sample


,n,Mean,Median,Volatility (monthly),Skewness,Excess kurtosis,5% quantile,CVaR 5% (mean of worst 5%),Worst month,% months < -5%,% months < -10%,Sharpe of the regime (ann.)
Risk-on (P > SMA200),295,+0.90%,+1.28%,3.50%,-0.66,1.09,-5.60%,-7.65%,-14.44%,6.1%,0.3%,+0.89
Risk-off (P < SMA200),98,+0.36%,+1.02%,6.52%,-0.37,-0.41,-10.07%,-13.37%,-16.54%,22.4%,5.1%,+0.19


  volatility ratio risk-off / risk-on = 1.86  (95% block-bootstrap CI 1.52 - 2.22)

Long S&P 500 TR sample


,n,Mean,Median,Volatility (monthly),Skewness,Excess kurtosis,5% quantile,CVaR 5% (mean of worst 5%),Worst month,% months < -5%,% months < -10%,Sharpe of the regime (ann.)
Risk-on (P > SMA200),766,+0.91%,+1.22%,3.74%,-0.67,2.35,-5.25%,-7.87%,-22.17%,5.7%,0.7%,+0.84
Risk-off (P < SMA200),344,+0.27%,+0.42%,5.96%,-0.22,1.91,-9.40%,-13.34%,-23.86%,14.8%,3.8%,+0.15


  volatility ratio risk-off / risk-on = 1.59  (95% block-bootstrap CI 1.35 - 1.85)



## 3. Robustness across time (SMA200 and monthly rebalancing fixed)

Long S&P 500 total-return sample, split by decision date. 1934-1949 is added because the data allow it; T-bill data start in 1934. The same split for the SPY sample is shown below it.

In [12]:
PERIODS = {
    "1934-1949": ("1934-01-01", "1949-12-31"),
    "1950-1969": ("1950-01-01", "1969-12-31"),
    "1970-1989": ("1970-01-01", "1989-12-31"),
    "1990-2009": ("1990-01-01", "2009-12-31"),
    "2010-present": ("2010-01-01", None),
    "1950-present": ("1950-01-01", None),
    "Full sample": (None, None),
}


def subperiod_table(pan: pd.DataFrame, periods: dict, eq_label: str) -> pd.DataFrame:
    rows = []
    for name, (a, b) in periods.items():
        sl = pan.loc[a:b]
        if len(sl) < 24:
            continue
        for reg in REGIMES:
            g = sl[sl["regime"] == reg]
            if len(g) < 2:
                continue
            se = g["excess"].std() / math.sqrt(len(g))
            rows.append({
                "period": name, "regime": reg.split(" ")[0], "n": len(g),
                f"{eq_label} return": g["r_eq"].mean(), "T-bill return": g["r_tbill"].mean(),
                "Excess return": g["excess"].mean(), "t-stat": g["excess"].mean() / se, "se": se,
                f"{eq_label} vol (ann.)": g["r_eq"].std() * math.sqrt(12),
                f"% months beats T-bills": (g["excess"] > 0).mean(),
            })
        on, off = sl.loc[sl["state"] == 1, "excess"], sl.loc[sl["state"] == 0, "excess"]
        if len(off) > 1:
            w = stats.ttest_ind(on, off, equal_var=False)
            rows.append({"period": name, "regime": "on - off", "n": len(sl), "Excess return": on.mean() - off.mean(),
                         "t-stat": w.statistic})
    return pd.DataFrame(rows).set_index(["period", "regime"])


SUB_FMT = {"S&P 500 return": "{:+.2%}", "SPY return": "{:+.2%}", "T-bill return": "{:.2%}", "Excess return": "{:+.2%}",
           "t-stat": "{:+.2f}", "S&P 500 vol (ann.)": "{:.1%}", "SPY vol (ann.)": "{:.1%}",
           "% months beats T-bills": "{:.0%}", "n": "{:.0f}"}
sub_idx = subperiod_table(pan_idx, PERIODS, "S&P 500")
print("Long S&P 500 TR sample -- monthly means; t-stat for the mean excess return (iid), 'on - off' row = Welch t")
display(sub_idx.drop(columns="se").style.format(SUB_FMT, na_rep=""))
sub_spy = subperiod_table(pan_spy, {"1993-2009": (None, "2009-12-31"), "2010-present": ("2010-01-01", None),
                                    "Full SPY sample": (None, None)}, "SPY")
print("SPY sample")
display(sub_spy.drop(columns="se").style.format(SUB_FMT, na_rep=""))

Long S&P 500 TR sample -- monthly means; t-stat for the mean excess return (iid), 'on - off' row = Welch t


SPY sample


In [13]:
sp = sub_idx.reset_index()
sp = sp[sp["regime"].isin(["Risk-on", "Risk-off"]) & ~sp["period"].isin(["1950-present", "Full sample"])]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Mean next-month excess return (annualised x12, 95% CI)",
                                                    "S&P 500 volatility (annualised)"))
for reg, col in (("Risk-on", C_ON), ("Risk-off", C_OFF)):
    d = sp[sp["regime"] == reg]
    fig.add_trace(go.Bar(x=d["period"], y=d["Excess return"] * 12, name=reg, marker_color=col,
                         error_y=dict(type="data", array=1.96 * d["se"] * 12)), row=1, col=1)
    fig.add_trace(go.Bar(x=d["period"], y=d["S&P 500 vol (ann.)"], name=reg, marker_color=col, showlegend=False),
                  row=1, col=2)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(barmode="group", height=450, title="Sub-periods, long S&P 500 TR sample")
fig.show()

### Signal strength: distance from the SMA200

The regime is a binary cut of a continuous variable, the distance $P/SMA_{200}-1$ at the decision date. Quintiles of that distance show whether the information is in the **mean** (a monotone pattern) or in the **risk**.

In [14]:
q = pan_idx.assign(bucket=pd.qcut(pan_idx["dist"], 5, labels=[f"Q{i}" for i in range(1, 6)]))
qt = q.groupby("bucket", observed=True).agg(
    dist_range=("dist", lambda s: f"{s.min():+.1%} .. {s.max():+.1%}"), n=("excess", "size"),
    share_risk_on=("state", "mean"), mean_excess=("excess", "mean"), vol=("excess", "std"),
    beats=("excess", lambda s: (s > 0).mean()), worst=("excess", "min"),
)
qt["t-stat"] = qt["mean_excess"] / (qt["vol"] / np.sqrt(qt["n"]))
display(qt.style.format({"share_risk_on": "{:.0%}", "mean_excess": "{:+.2%}", "vol": "{:.2%}", "beats": "{:.0%}",
                         "worst": "{:+.1%}", "t-stat": "{:+.2f}"}))
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=qt.index.astype(str), y=qt["mean_excess"], name="mean excess (monthly)",
                     marker_color=[C_OFF, C_OFF, C_MIX, C_ON, C_ON]), secondary_y=False)
fig.add_trace(go.Scatter(x=qt.index.astype(str), y=qt["vol"], name="volatility (monthly)", mode="lines+markers",
                         line=dict(color="black")), secondary_y=True)
fig.update_yaxes(tickformat=".1%", secondary_y=False, title_text="mean excess")
fig.update_yaxes(tickformat=".1%", secondary_y=True, title_text="volatility")
fig.update_layout(title="Next-month excess return and volatility by quintile of distance from SMA200 (Q1 = furthest below)",
                  height=420)
fig.show()

,dist_range,n,share_risk_on,mean_excess,vol,beats,worst,t-stat
bucket,,,,,,,,
Q1,-34.1% .. -3.7%,222,0%,+0.64%,6.30%,54%,-21.7%,+1.51
Q2,-3.6% .. +2.1%,222,45%,+0.11%,4.87%,57%,-23.9%,+0.33
Q3,+2.2% .. +5.4%,222,100%,+1.04%,3.49%,64%,-11.0%,+4.45
Q4,+5.4% .. +9.4%,222,100%,+0.92%,3.96%,63%,-22.2%,+3.45
Q5,+9.5% .. +23.9%,222,100%,+0.84%,3.46%,64%,-8.8%,+3.64


## 4. Return shift (A) or tail avoidance (B)?

Three diagnostics:

* **Where do the worst months fall?** If risk-off months hold a disproportionate share of the worst equity months, the regime identifies *risk*.
* **Cumulative contribution**: the total excess return buy & hold earned in risk-on months versus risk-off months over time.
* **Excess-return accounting**: buy & hold's mean excess $= p_{on}\,m_{on} + p_{off}\,m_{off}$. The strategy keeps $p_{on}\,m_{on}$ and replaces $p_{off}\,m_{off}$ with zero excess (T-bills). Its *return* can only rise if $m_{off} < 0$.

In [15]:
def worst_month_table(pan: pd.DataFrame) -> pd.DataFrame:
    p_off = (pan["state"] == 0).mean()
    srt = pan.sort_values("excess")
    rows = {}
    for k in (10, 20, 50, int(round(0.05 * len(pan)))):
        w = srt.head(k)
        rows[f"worst {k} months"] = {
            "in risk-off": int((w["state"] == 0).sum()),
            "share in risk-off": (w["state"] == 0).mean(),
            "unconditional risk-off share": p_off,
            "lift (x)": (w["state"] == 0).mean() / p_off,
            "mean excess of these months": w["excess"].mean(),
        }
    best = srt.tail(20)
    rows["best 20 months"] = {"in risk-off": int((best["state"] == 0).sum()), "share in risk-off": (best["state"] == 0).mean(),
                              "unconditional risk-off share": p_off, "lift (x)": (best["state"] == 0).mean() / p_off,
                              "mean excess of these months": best["excess"].mean()}
    return pd.DataFrame(rows).T


def accounting(pan: pd.DataFrame) -> pd.Series:
    on, off = pan["state"] == 1, pan["state"] == 0
    p_on, p_off = on.mean(), off.mean()
    m_on, m_off = pan.loc[on, "excess"].mean(), pan.loc[off, "excess"].mean()
    return pd.Series({
        "share of months risk-on": p_on, "share of months risk-off": p_off,
        "mean excess, risk-on months (ann.)": 12 * m_on, "mean excess, risk-off months (ann.)": 12 * m_off,
        "B&H mean excess (ann.)": 12 * (p_on * m_on + p_off * m_off),
        "  from risk-on months": 12 * p_on * m_on, "  from risk-off months": 12 * p_off * m_off,
        "SMA200 mean excess before costs (ann.)": 12 * p_on * m_on,
        "excess return given up by stepping aside (ann.)": 12 * p_off * m_off,
    })


WM_FMT = {"share in risk-off": "{:.0%}", "unconditional risk-off share": "{:.0%}", "lift (x)": "{:.2f}",
          "mean excess of these months": "{:+.2%}", "in risk-off": "{:.0f}"}
for lab, pan in (("SPY sample", pan_spy), ("Long S&P 500 TR sample", pan_idx)):
    print(lab)
    display(worst_month_table(pan).style.format(WM_FMT))
acc = pd.DataFrame({"SPY sample": accounting(pan_spy), "Long S&P 500 TR": accounting(pan_idx)})
display(acc.style.format("{:+.2%}"))

SPY sample


,in risk-off,share in risk-off,unconditional risk-off share,lift (x),mean excess of these months
worst 10 months,9,90%,25%,3.61,-12.01%
worst 20 months,15,75%,25%,3.01,-10.25%
worst 50 months,26,52%,25%,2.09,-7.52%
best 20 months,16,80%,25%,3.21,+9.16%


Long S&P 500 TR sample


,in risk-off,share in risk-off,unconditional risk-off share,lift (x),mean excess of these months
worst 10 months,8,80%,31%,2.58,-17.58%
worst 20 months,15,75%,31%,2.42,-14.12%
worst 50 months,32,64%,31%,2.07,-10.70%
worst 56 months,35,62%,31%,2.02,-10.28%
best 20 months,15,75%,31%,2.42,+12.58%


,SPY sample,Long S&P 500 TR
share of months risk-on,+75.06%,+69.01%
share of months risk-off,+24.94%,+30.99%
"mean excess, risk-on months (ann.)",+10.77%,+10.93%
"mean excess, risk-off months (ann.)",+4.34%,+3.18%
B&H mean excess (ann.),+9.17%,+8.53%
from risk-on months,+8.08%,+7.54%
from risk-off months,+1.08%,+0.99%
SMA200 mean excess before costs (ann.),+8.08%,+7.54%
excess return given up by stepping aside (ann.),+1.08%,+0.99%


In [16]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.06,
                    subplot_titles=("Cumulative log excess return of buy & hold, split by the regime at each decision",
                                    "Next-month excess return by regime"))
for reg, col in zip(REGIMES, [C_ON, C_OFF]):
    contrib = np.log1p(pan_idx["excess"]).where(pan_idx["regime"] == reg, 0.0).cumsum()
    fig.add_trace(go.Scatter(x=pan_idx.index, y=contrib, name=f"earned in {reg} months", line=dict(color=col)), row=1, col=1)
    g = pan_idx[pan_idx["regime"] == reg]
    fig.add_trace(go.Scatter(x=g.index, y=g["excess"], mode="markers", marker=dict(color=col, size=4),
                             name=reg, showlegend=False), row=2, col=1)
fig.update_yaxes(title_text="sum of log excess", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(height=750, hovermode="x unified", legend=dict(orientation="h", y=-0.08))
fig.show()

## 5. Where does the higher Sharpe ratio come from?

The strategies are compared on identical daily rows. Sharpe and Sortino use returns in excess of the same T-bill series.

* **SMA200 net**: the strategy (5 bp per switch).
* **SMA200 gross**: no costs.
* **SMA200, cash earns 0**: the same switches, but the risk-off months earn nothing. This isolates the **T-bill income**.
* **Constant mix**: a constant equity weight equal to the strategy's average time in equities, with the rest in T-bills, rebalanced daily. It has the same average exposure with **no timing**. If timing added nothing, SMA200 would look like this mix.

In [17]:
def compare(bt: pd.DataFrame, eq_label: str) -> pd.DataFrame:
    w = bt["pos"].mean()
    zero_cash = bt["pos"] * bt["r_eq"] - bt["cost"]
    mix = w * bt["r_eq"] + (1 - w) * bt["r_cash"]
    ones, zeros = pd.Series(1.0, index=bt.index), pd.Series(0.0, index=bt.index)
    cols = {
        f"Buy & hold {eq_label}": perf_stats(bt["r_bh"], bt["r_cash"], ones, zeros),
        "SMA200 monthly (net)": perf_stats(bt["r_strat"], bt["r_cash"], bt["pos"], bt["turnover"]),
        "SMA200 monthly (gross)": perf_stats(bt["r_gross"], bt["r_cash"], bt["pos"], bt["turnover"]),
        "SMA200, cash earns 0": perf_stats(zero_cash, bt["r_cash"], bt["pos"], bt["turnover"]),
        f"Constant mix {w:.0%} / {1 - w:.0%}": perf_stats(mix, bt["r_cash"], pd.Series(w, index=bt.index), zeros),
    }
    return pd.DataFrame(cols)


def show_compare(tab: pd.DataFrame, title: str):
    fmt = tab.astype(object).copy()
    for r in tab.index:
        if r in ("Sharpe", "Sortino", "Calmar", "Switches / yr"):
            fmt.loc[r] = [f"{v:.2f}" for v in tab.loc[r]]
        elif r == "Switches":
            fmt.loc[r] = [f"{int(v)}" for v in tab.loc[r]]
        else:
            fmt.loc[r] = [pct(v) for v in tab.loc[r]]
    print(title)
    display(fmt)


bt_user = backtest(px_spy, cash_spy["tr_open"], SMA_WINDOW, COST_BPS, "next_open", start=USER_START, rebalance=REBALANCE)
bt_long = bt_idx.loc[LONG_START:]
cmp_spy, cmp_user, cmp_long = compare(bt_spy, "SPY"), compare(bt_user, "SPY"), compare(bt_long, "S&P 500")
show_compare(cmp_spy, f"SPY, full history {bt_spy.index[0].date()} -> {bt_spy.index[-1].date()} (cash={CASH_SOURCE})")
show_compare(cmp_user, f"SPY, your backtest window {bt_user.index[0].date()} -> {bt_user.index[-1].date()}")
show_compare(cmp_long, f"S&P 500 TR, {bt_long.index[0].date()} -> {bt_long.index[-1].date()} (next-close execution, no fund fees)")

SPY, full history 1993-11-15 -> 2026-10-19 (cash=bil)


,Buy & hold SPY,SMA200 monthly (net),SMA200 monthly (gross),"SMA200, cash earns 0",Constant mix 75% / 25%
CAGR,10.84%,10.28%,10.35%,9.73%,9.05%
Arithmetic return (ann.),12.04%,10.60%,10.67%,10.10%,9.65%
Excess return (ann.),9.65%,8.22%,8.28%,7.71%,7.26%
Volatility (ann.),18.58%,12.66%,12.66%,12.65%,13.97%
Sharpe,0.52,0.65,0.65,0.61,0.52
Sortino,0.73,0.91,0.91,0.85,0.73
Max drawdown,-55.42%,-25.80%,-25.57%,-26.77%,-44.13%
Calmar,0.20,0.40,0.40,0.36,0.20
% time in equities,100.00%,75.26%,75.26%,75.26%,75.26%
% time in T-bills,0.00%,24.74%,24.74%,24.74%,24.74%


SPY, your backtest window 2007-03-15 -> 2026-10-19


,Buy & hold SPY,SMA200 monthly (net),SMA200 monthly (gross),"SMA200, cash earns 0",Constant mix 78% / 22%
CAGR,11.15%,9.62%,9.70%,9.31%,9.24%
Arithmetic return (ann.),12.40%,9.97%,10.04%,9.68%,9.94%
Excess return (ann.),10.97%,8.54%,8.61%,8.25%,8.51%
Volatility (ann.),19.00%,12.37%,12.37%,12.35%,14.71%
Sharpe,0.57,0.69,0.70,0.67,0.57
Sortino,0.80,0.96,0.97,0.93,0.80
Max drawdown,-55.42%,-25.80%,-25.57%,-26.77%,-45.24%
Calmar,0.20,0.37,0.38,0.35,0.20
% time in equities,100.00%,77.51%,77.51%,77.51%,77.51%
% time in T-bills,0.00%,22.49%,22.49%,22.49%,22.49%


S&P 500 TR, 1934-02-02 -> 2026-10-19 (next-close execution, no fund fees)


,Buy & hold S&P 500,SMA200 monthly (net),SMA200 monthly (gross),"SMA200, cash earns 0",Constant mix 69% / 31%
CAGR,11.05%,10.61%,10.69%,9.41%,8.93%
Arithmetic return (ann.),11.92%,10.80%,10.88%,9.71%,9.25%
Excess return (ann.),8.59%,7.48%,7.55%,6.38%,5.93%
Volatility (ann.),16.73%,11.70%,11.70%,11.70%,11.54%
Sharpe,0.51,0.64,0.65,0.54,0.51
Sortino,0.72,0.88,0.89,0.75,0.72
Max drawdown,-55.25%,-32.96%,-32.96%,-32.96%,-40.90%
Calmar,0.20,0.32,0.32,0.29,0.22
% time in equities,100.00%,68.97%,68.97%,68.97%,68.97%
% time in T-bills,0.00%,31.03%,31.03%,31.03%,31.03%


In [18]:
def sharpe_decomposition(tab: pd.DataFrame) -> pd.Series:
    bh, sm = tab.iloc[:, 0], tab["SMA200 monthly (net)"]
    ret_term = math.log(sm["Excess return (ann.)"] / bh["Excess return (ann.)"])
    vol_term = -math.log(sm["Volatility (ann.)"] / bh["Volatility (ann.)"])
    return pd.Series({
        "Sharpe buy & hold": bh["Sharpe"], "Sharpe SMA200": sm["Sharpe"],
        "excess return B&H": bh["Excess return (ann.)"], "excess return SMA200": sm["Excess return (ann.)"],
        "vol B&H": bh["Volatility (ann.)"], "vol SMA200": sm["Volatility (ann.)"],
        "ln(SR ratio)": math.log(sm["Sharpe"] / bh["Sharpe"]),
        "= return term ln(mu_s/mu_bh)": ret_term, "+ volatility term -ln(sig_s/sig_bh)": vol_term,
        "max DD B&H": bh["Max drawdown"], "max DD SMA200": sm["Max drawdown"],
        "CAGR gain from T-bill income": sm["CAGR"] - tab["SMA200, cash earns 0"]["CAGR"],
    })


dec = pd.DataFrame({"SPY full": sharpe_decomposition(cmp_spy), "SPY your window": sharpe_decomposition(cmp_user),
                    "S&P 500 TR long": sharpe_decomposition(cmp_long)})
print("Sharpe decomposition: ln(SR_s/SR_bh) = ln(excess-return ratio) - ln(volatility ratio)   "
      "(approximate: Sharpe uses the volatility of excess returns)")
display(dec.style.format(lambda v: f"{v:+.3f}" if abs(v) < 5 else f"{v:.2f}"))

Sharpe decomposition: ln(SR_s/SR_bh) = ln(excess-return ratio) - ln(volatility ratio)   (approximate: Sharpe uses the volatility of excess returns)

,SPY full,SPY your window,S&P 500 TR long
Sharpe buy & hold,+0.518,+0.574,+0.513
Sharpe SMA200,+0.649,+0.690,+0.639
excess return B&H,+0.097,+0.110,+0.086
excess return SMA200,+0.082,+0.085,+0.075
vol B&H,+0.186,+0.190,+0.167
vol SMA200,+0.127,+0.124,+0.117
ln(SR ratio),+0.226,+0.184,+0.219
= return term ln(mu_s/mu_bh),-0.161,-0.251,-0.139
+ volatility term -ln(sig_s/sig_bh),+0.384,+0.429,+0.358
max DD B&H,-0.554,-0.554,-0.553


In [19]:
curves = pd.DataFrame({
    "Buy & hold SPY": (1 + bt_spy["r_bh"]).cumprod(),
    "SMA200 monthly (net)": (1 + bt_spy["r_strat"]).cumprod(),
    f"Constant mix {bt_spy['pos'].mean():.0%}": (1 + bt_spy["pos"].mean() * bt_spy["r_eq"]
                                                 + (1 - bt_spy["pos"].mean()) * bt_spy["r_cash"]).cumprod(),
})
dd = curves / curves.cummax() - 1
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.55, 0.15, 0.3], vertical_spacing=0.05,
                    subplot_titles=("Growth of $1 (log scale), SPY full history", "SMA200 position (1 = SPY, 0 = T-bills)",
                                    "Drawdown"))
for (name, s), col in zip(curves.items(), [C_BH, C_STRAT, C_MIX]):
    fig.add_trace(go.Scatter(x=s.index, y=s, name=name, line=dict(color=col, width=1.3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=dd.index, y=dd[name], name=name, line=dict(color=col, width=1), showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=bt_spy.index, y=bt_spy["pos"], line_shape="hv", fill="tozeroy", name="in SPY",
                         line=dict(color=C_STRAT, width=0.5), showlegend=False), row=2, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=3, col=1)
fig.update_layout(height=850, hovermode="x unified", legend=dict(orientation="h", y=-0.06))
fig.show()

### Is the advantage concentrated in a few crash months?

The monthly Sharpe ratios of buy & hold and SMA200 (gross, from the decision panel) are recomputed after removing the *K* worst buy-&-hold months from **both** series.
If the SMA200 advantage disappears once a handful of crash months are removed, the edge is tail avoidance (B), not a broad shift in expected returns.

In [20]:
def tail_exclusion(pan: pd.DataFrame, ks=(0, 5, 10, 20, 40)) -> pd.DataFrame:
    rows = {}
    order = pan["excess"].sort_values().index
    for k in ks:
        sl = pan.drop(order[:k])
        bh, sm = sl["excess"], sl["excess"] * sl["state"]
        rows[f"drop worst {k}"] = {
            "Sharpe B&H": bh.mean() / bh.std() * math.sqrt(12), "Sharpe SMA200": sm.mean() / sm.std() * math.sqrt(12),
            "mean excess B&H (ann.)": 12 * bh.mean(), "mean excess SMA200 (ann.)": 12 * sm.mean(),
            "vol B&H (ann.)": bh.std() * math.sqrt(12), "vol SMA200 (ann.)": sm.std() * math.sqrt(12),
            "of dropped: in risk-off": int((pan.loc[order[:k], "state"] == 0).sum()),
        }
    out = pd.DataFrame(rows).T
    out["Sharpe gap"] = out["Sharpe SMA200"] - out["Sharpe B&H"]
    return out


TE_FMT = {"Sharpe B&H": "{:.2f}", "Sharpe SMA200": "{:.2f}", "Sharpe gap": "{:+.2f}", "of dropped: in risk-off": "{:.0f}",
          "mean excess B&H (ann.)": "{:+.2%}", "mean excess SMA200 (ann.)": "{:+.2%}", "vol B&H (ann.)": "{:.1%}",
          "vol SMA200 (ann.)": "{:.1%}"}
for lab, pan in (("SPY sample", pan_spy), ("Long S&P 500 TR sample", pan_idx)):
    print(lab)
    display(tail_exclusion(pan).style.format(TE_FMT))

SPY sample


,Sharpe B&H,Sharpe SMA200,mean excess B&H (ann.),mean excess SMA200 (ann.),vol B&H (ann.),vol SMA200 (ann.),of dropped: in risk-off,Sharpe gap
drop worst 0,0.60,0.76,+9.17%,+8.08%,15.4%,10.6%,0,+0.17
drop worst 5,0.80,0.84,+11.45%,+8.64%,14.3%,10.3%,4,+0.04
drop worst 10,0.96,0.84,+13.17%,+8.75%,13.7%,10.4%,9,-0.12
drop worst 20,1.27,1.01,+16.25%,+10.09%,12.8%,9.9%,15,-0.26
drop worst 40,1.89,1.50,+21.53%,+13.44%,11.4%,9.0%,22,-0.40


Long S&P 500 TR sample


,Sharpe B&H,Sharpe SMA200,mean excess B&H (ann.),mean excess SMA200 (ann.),vol B&H (ann.),vol SMA200 (ann.),of dropped: in risk-off,Sharpe gap
drop worst 0,0.54,0.69,+8.53%,+7.54%,15.8%,10.9%,0,+0.15
drop worst 5,0.65,0.74,+9.70%,+7.82%,15.0%,10.6%,4,+0.09
drop worst 10,0.72,0.77,+10.52%,+8.03%,14.6%,10.5%,8,+0.04
drop worst 20,0.84,0.82,+11.79%,+8.46%,14.1%,10.3%,15,-0.02
drop worst 40,1.04,0.94,+14.01%,+9.44%,13.4%,10.0%,27,-0.10


### Placebo test: does the *timing* matter, or just being out ~30% of the time?

The regime series is shifted in a circle by every offset of at least 12 months. Each shift keeps the share of risk-off months and the length of the risk-off spells, but breaks the alignment with returns.
If the true signal's Sharpe ratio is not unusually high among these placebo signals, the improvement comes from exposure, not timing information.

In [21]:
def placebo(pan: pd.DataFrame, min_shift=12):
    ex, s = pan["excess"].to_numpy(), pan["state"].to_numpy()
    n = len(ex)
    shifts = np.arange(min_shift, n - min_shift + 1)
    S = np.stack([np.roll(s, k) for k in shifts])
    r = S * ex
    sr = r.mean(1) / r.std(1, ddof=1) * math.sqrt(12)
    mu, vol = 12 * r.mean(1), r.std(1, ddof=1) * math.sqrt(12)
    act = s * ex
    actual = {"Sharpe": act.mean() / act.std(ddof=1) * math.sqrt(12), "mean": 12 * act.mean(),
              "vol": act.std(ddof=1) * math.sqrt(12)}
    p = {"Sharpe": (sr >= actual["Sharpe"]).mean(), "mean": (mu >= actual["mean"]).mean(), "vol": (vol <= actual["vol"]).mean()}
    return sr, mu, vol, actual, p


plc = {}
for lab, pan in (("SPY", pan_spy), ("S&P 500 long", pan_idx)):
    sr, mu, vol, actual, p = placebo(pan)
    plc[lab] = (sr, actual)
    bh = pan["excess"]
    print(f"{lab}: actual SMA200 Sharpe {actual['Sharpe']:.2f} vs placebo median {np.median(sr):.2f} "
          f"(B&H {bh.mean() / bh.std() * math.sqrt(12):.2f}); share of placebos with Sharpe >= actual: p = {p['Sharpe']:.3f}")
    print(f"   mean excess: actual {actual['mean']:+.2%} vs placebo median {np.median(mu):+.2%} (p = {p['mean']:.3f}); "
          f"volatility: actual {actual['vol']:.1%} vs placebo median {np.median(vol):.1%} (share of placebos with lower vol: {p['vol']:.3f})")

fig = go.Figure()
for (lab, (sr, actual)), col in zip(plc.items(), [C_STRAT, C_BH]):
    fig.add_trace(go.Histogram(x=sr, name=f"{lab} placebo shifts", opacity=0.55, marker_color=col, nbinsx=40))
    fig.add_vline(x=actual["Sharpe"], line_color=col, line_width=3, annotation_text=f"{lab} actual",
                  annotation_position="top")
fig.update_layout(barmode="overlay", title="Sharpe ratio of the SMA200 signal vs circularly shifted placebo signals",
                  xaxis_title="annualised Sharpe (monthly panel, before costs)", height=420)
fig.show()

SPY: actual SMA200 Sharpe 0.76 vs placebo median 0.50 (B&H 0.60); share of placebos with Sharpe >= actual: p = 0.005
   mean excess: actual +8.08% vs placebo median +6.63% (p = 0.108); volatility: actual 10.6% vs placebo median 13.5% (share of placebos with lower vol: 0.000)
S&P 500 long: actual SMA200 Sharpe 0.69 vs placebo median 0.44 (B&H 0.54); share of placebos with Sharpe >= actual: p = 0.000
   mean excess: actual +7.54% vs placebo median +5.79% (p = 0.022); volatility: actual 10.9% vs placebo median 13.2% (share of placebos with lower vol: 0.000)


## 6. Evidence summary (computed)

In [22]:
def verdict(inf, pan):
    on, off, diff = inf.loc[REGIMES[0]], inf.loc[REGIMES[1]], inf.loc["Difference (on - off)"]
    vol_r, (lo, hi) = vol_ratio_ci(pan)
    return {
        "H1 risk-on mean excess > 0": f"{on['Mean excess (monthly)']:+.2%}/month, NW t = {on['t-stat (Newey-West)']:+.2f} "
                                      f"-> {'supported' if on['t-stat (Newey-West)'] > 1.96 else 'not significant'}",
        "H2 risk-off mean excess < 0": f"{off['Mean excess (monthly)']:+.2%}/month, NW t = {off['t-stat (Newey-West)']:+.2f} "
                                       f"-> {'supported' if off['t-stat (Newey-West)'] < -1.96 else 'NOT supported'}",
        "on - off difference": f"{diff['Mean excess (monthly)']:+.2%}/month, NW t = {diff['t-stat (Newey-West)']:+.2f}, "
                               f"bootstrap one-sided p = {diff['One-sided p (bootstrap)']:.3f}",
        "risk-off / risk-on volatility": f"{vol_r:.2f}x (95% CI {lo:.2f}-{hi:.2f})",
    }


display(pd.DataFrame({"SPY sample": verdict(inf_spy, pan_spy), "Long S&P 500 TR": verdict(inf_idx, pan_idx)}))

,SPY sample,Long S&P 500 TR
H1 risk-on mean excess > 0,"+0.90%/month, NW t = +4.97 -> supported","+0.91%/month, NW t = +6.72 -> supported"
H2 risk-off mean excess < 0,"+0.36%/month, NW t = +0.54 -> NOT supported","+0.27%/month, NW t = +0.90 -> NOT supported"
on - off difference,"+0.54%/month, NW t = +0.77, bootstrap one-side...","+0.65%/month, NW t = +2.01, bootstrap one-side..."
risk-off / risk-on volatility,1.86x (95% CI 1.51-2.22),1.59x (95% CI 1.35-1.85)


## 7. Interpretation

*Figures quoted here are from the run on data to 2026-10-15. Re-running updates every table above, but not this text.*

**1. Information about the next month's excess return.**
H1 holds clearly. In risk-on months SPY beat T-bills by +0.90 %/month (Newey-West t ~ 5.0; S&P 500 1934+: +0.91 %, t ~ 6.7).
H2 is **rejected**. In risk-off months the mean excess return was still **positive**: +0.36 %/month for SPY and +0.27 %/month for the long sample, with t-stats around +0.5 to +0.9 and bootstrap CIs straddling zero.
The on-minus-off gap (~0.55-0.65 %/month) is only marginally significant in the long sample (NW t ~ 2.0, bootstrap p ~ 0.03) and insignificant in the SPY sample. Being below the SMA200 has **not** identified periods in which the equity risk premium turns negative. It identified periods in which it was **smaller and much noisier**.

**2. Sub-periods.**
The risk-off mean excess was negative in only two of five sub-periods (1970-89 and 1990-2009), and never significantly so. In 2010-present it was strongly *positive* (+2.4 %/month), driven by fast V-shaped rebounds such as 2020. In contrast, risk-off volatility is higher than risk-on volatility in **every** sub-period. The mean effect is unstable; the volatility effect is robust.

**3. What the regime really identifies.**
Risk-off months have 1.6-1.9 times the volatility of risk-on months (bootstrap CIs exclude 1). They contain 75-90 % of the 10-20 worst months, against a 25-31 % base rate, but **also 75-80 % of the 20 best months**. The distance-from-SMA quintiles are not monotone: the deepest-below quintile still averaged +0.6 %/month. The regime is a **volatility / turbulence classifier**, not a return-sign forecaster. The mean-variance trade-off is what differs: the regime Sharpe ratio is ~0.85-0.9 in risk-on months against ~0.15-0.2 in risk-off months.

**4. Source of the higher Sharpe ratio.**
In every sample the Sharpe gain (log ratio +0.18 to +0.23) comes **entirely from lower volatility**. The volatility term is +0.36 to +0.43, while the return term is *negative* (-0.14 to -0.25), because the strategy gives up ~1 %/year of excess return by stepping aside in months that were positive on average.
The maximum drawdown halves (-55 % to -26 % for SPY), so Calmar roughly doubles. T-bill income while out adds 0.3-1.2 %/year of CAGR and 0.02-0.10 of Sharpe (the most in the long sample with its higher rates), which is helpful but not the main source.
The constant mix with the same average equity exposure has the *same* Sharpe as buy & hold, so the gain is **not** just from holding less equity. In the placebo test the actual timing beats >99 % of shifted signals on Sharpe. That shows the timing is informative, overwhelmingly through volatility; the actual signal has lower volatility than every placebo.

**5. Fragility.**
Removing the 5-10 worst buy-&-hold months (SPY sample), or ~20 (long sample), from both series erases the Sharpe advantage. Most of those months fall in the risk-off regime. The edge rests on a small number of crash episodes: the 1930s-40s, 1973-74, 2000-02, 2008-09 and 2020.

**Verdict: mostly (B), with a weak (A) component.** SMA200 does not reliably forecast a negative equity premium. Its main value is moving out of the high-volatility, fat-tailed regime, where the premium per unit of risk is poor. That cuts portfolio volatility and drawdowns by far more than it costs in return.

## 8. Conclusion

There is an **economic justification, but a narrower one than "SMA200 predicts bear markets"**. Over ninety years of S&P 500 data and the full SPY history:

* below the SMA200, equity excess returns were on average still positive but small and statistically indistinguishable from zero, while volatility roughly doubled and the worst months clustered there;
* stepping into T-bills therefore **sacrifices a little expected return** (~1 %/year) in exchange for **~30-35 % lower volatility and a 40-55 % smaller maximum drawdown**. This is a better risk/return trade-off than a static mix with the same average exposure, and the placebo test says the timing, not the exposure level, delivers it;
* the improvement is concentrated in a handful of crashes and is not stable in every sub-period. 2010-present is a counter-example, where risk-off months were the best months to hold equities.

The SMA200/T-bill rule is best understood as a **crude volatility- and tail-risk-management overlay**. It is a reasonable tool when the goal is smaller drawdowns and a smoother path. It should not be expected to raise long-run returns, and it will underperform in markets with sharp V-shaped recoveries. Taxes, trading at the open and a different month-end anchor would all reduce the margin.